# hl

> Syntax colouring for code and for unified diffs, in the terminal or as HTML.

When the agent edits a file, it shows a unified diff. The diff marks removed lines with `-` and added lines with `+`, and groups them into hunks under `@@` headers. This module colours that diff and colours plain code. It uses [fastpylight](https://github.com/AnswerDotAI/fastpylight) to tokenize the source. A token is one piece of code, such as a keyword or a string. Its scope is the name of what it is, such as `string` or `function.builtin`. A theme maps scopes to colours. fastpylight's themes are the Lumis themes.

The functions give three kinds of output.

| Output | Functions | Use |
|---|---|---|
| `rich.Text` | `hl_text`, `diff_rich` | The terminal UI draws it. |
| Tokens as plain data | `hl_toks`, `hunk_toks`, `diff_toks` | A surface that does its own drawing. |
| HTML | `diff_html` | Style it with a stylesheet from `fastpylight.theme_css`. |

The module keeps no theme of its own. `hl_text` and `diff_rich` take a theme name, and `diff_rich` also takes a palette, a dict of colours. `ramabana.cli` calls `diff_rich` with the palette in use. The token functions and `diff_html` take no colours.

In [ ]:
#| default_exp hl

In [ ]:
#| hide
from fastcore.test import test_eq

In [ ]:
#| export
import functools, html, re
import fastpylight
from fastcore.basics import ifnone
from rich.text import Text

## Code

`hl_text(code, lang, theme)` colours a piece of code for the terminal and returns a `rich.Text`. `theme` is a Lumis theme name. `HL_THEMES` maps each palette name the terminal accepts to a Lumis theme name. For example, `--theme nord` colours code with the Lumis theme `nord`. `hl_text` returns `code` unstyled when `lang` is `'plaintext'`.

`scope_style(scope, theme)` returns the rich style a theme gives one scope. It returns an empty string when the theme gives the scope no foreground colour. When the theme does not name a scope, `scope_style` uses the longest dotted prefix of the scope that the theme does name. For example, `function.builtin` falls back to `function`.

In [ ]:
#| export
#: fastpylight (Lumis) themes, one per palette name
HL_THEMES = {'github-dark': 'github_dark', 'dark': 'github_dark', 'light': 'github_light',
             'gruvbox': 'gruvbox_dark', 'gruvbox-light': 'gruvbox_light', 'nord': 'nord',
             'dracula': 'dracula', 'solarized': 'solarized_osaka_dark',
             'solarized-light': 'solarized_osaka_light', 'tokyonight': 'tokyonight_night',
             'catppuccin': 'catppuccin_mocha', 'kanagawa': 'kanagawa_wave', 'everforest': 'everforest_dark',
             'latte': 'catppuccin_latte'}

@functools.cache
def _hl_colors(theme): return fastpylight.theme_colors(theme)

def scope_style(scope, theme='github_dark'):
    "The rich style Lumis `theme` gives token `scope`; the longest dotted prefix it names wins."
    c = _hl_colors(theme)
    while scope and scope not in c: scope = scope.rpartition('.')[0]
    if not (s := c.get(scope)) or not s['fg']: return ''
    return ' '.join([s['fg']] + ['bold'] * bool(s['bold']) + ['italic'] * bool(s['italic']))

def hl_text(code, lang, theme='github_dark'):
    "`code` as a rich `Text`, coloured by fastpylight with Lumis `theme`."
    t = Text(code)
    if lang == 'plaintext': return t
    code = t.plain
    at = [i for i, ch in enumerate(code) for _ in ch.encode()] + [len(code)]
    for a, b, k in fastpylight.tokenize(code, lang):
        if (st := scope_style(k, theme)): t.stylize(st, at[a], at[b])
    return t

fastpylight counts positions in UTF-8 bytes. rich counts characters. `hl_text` converts between them, and each coloured span starts on the character where its token starts. rich rewrites a line ending such as `\r\n` to `\n`. The spans stay on their tokens after the rewrite:

In [ ]:
t = hl_text('a\r\nb = 1  # one', 'python')
test_eq(t.plain, 'a\nb = 1  # one')
assert any(t.plain[s.start:s.end] == '1' for s in t.spans)
t

`hl_toks(code, lang)` returns the tokens as data instead: one list per line, each token a `(start, end, scope)` tuple in characters. Use it when you draw the code yourself. A token that runs across a line break, such as a triple-quoted string, is cut at the break and appears on both lines:

In [ ]:
#| export
def hl_toks(code, lang):
    "Per line of `code`, its tokens as `(start, end, scope)` in characters; a token crossing a line break is cut at it."
    ls = code.split('\n')
    out = [[] for _ in ls]
    if lang == 'plaintext' or not code: return out
    at = [i for i, ch in enumerate(code) for _ in ch.encode()] + [len(code)]
    starts = [0]
    for l in ls[:-1]: starts.append(starts[-1] + len(l) + 1)
    n = 0
    for a, b, k in fastpylight.tokenize(code, lang):
        a, b = at[a], at[b]
        while n + 1 < len(starts) and starts[n + 1] <= a: n += 1
        i = n
        while a < b and i < len(ls):
            if (end := min(b, starts[i] + len(ls[i]))) > a: out[i].append((a - starts[i], end - starts[i], k))
            i += 1
            if i < len(ls): a = max(a, starts[i])
    return out

In [ ]:
t = hl_toks('é = """a\nb"""', 'python')
test_eq(t[0][0], (0, 1, 'variable'))
test_eq([k for *_, k in t[1]], ['string'])

## Diffs

`diff_rich(s, palette, theme)` colours a unified diff for the terminal. It also colours a diffstat, the `file | 3 ++-` summary git prints, and returns any other text in gray. `palette` maps `fg1`, `gray`, `red`, `green`, `aqua` and `bg0` to colours. File headers are bold, hunk headers are `aqua`, and added and removed lines get a background tinted 15% toward green or red. The code inside each hunk is coloured with `theme` in its own language. The old side and the new side of a hunk are each tokenized whole, so a string that spans several lines keeps its colour on every one. `is_diff` says whether a text holds a unified diff at all.

The same split is available as data. `diff_toks` returns one entry per line of a diff: the line's tokens, or `None` for a line outside a hunk body, such as a header. `hunk_toks` does the same for one hunk body. `diff_html` renders each code line as HTML through `toks_html`, which wraps each token in a span whose classes are its scope and every prefix of it, as in `hl-function hl-function-builtin`, so a stylesheet that styles only `function` still colours it.

`diff_lang` picks the language of a hunk from the path on its `---` or `+++` line. A notebook's diff holds the source of its cells, so a path ending in `.ipynb` reads as Python. So does a single cell, written `nb.ipynb#id` the way `edit_cell` writes it. Any other path takes fastpylight's guess from the file name. The cells below check `diff_lang`, then `diff_html` on a small diff.

In [ ]:
#| export
_STAT = re.compile(r'^(\s*\S.*?\s\|\s+\d+\s)(\+*)(-*)\s*$')
_HUNK = re.compile(r'^@@ -\d+(?:,(\d+))? \+\d+(?:,(\d+))? @@')

def _mix(a, b, t): return '#' + ''.join(f'{round(int(a[i:i+2], 16) * (1 - t) + int(b[i:i+2], 16) * t):02x}' for i in (1, 3, 5))

def hunk_toks(ls, lang):
    "Tokens per hunk body line, relative to the line after its prefix; each side is tokenized whole, so a string spanning lines keeps its scope. `None` for a `\\` line."
    ls = [l or ' ' for l in ls]
    def side(keep): return iter(hl_toks('\n'.join(l[1:] for l in ls if l[:1] in keep), lang))
    old, new, out = side((' ', '-')), side((' ', '+')), []
    for l in ls:
        k = l[:1]
        if k == '\\': out.append(None); continue
        out.append(next(old) if k == '-' else next(new))
        if k == ' ': next(old)
    return out

def _hunk(ls, lang, palette, theme):
    "Rows for one hunk body, coloured from `hunk_toks`."
    G, ls, rows = palette, [l or ' ' for l in ls], []
    for l, toks in zip(ls, hunk_toks(ls, lang)):
        k = l[:1]
        c = {'+': G['green'], '-': G['red']}.get(k, G['gray'])
        if toks is None: rows.append(Text(l, style=c)); continue
        code = Text(l[1:])
        for a, b, s in toks:
            if (st := scope_style(s, theme)): code.stylize(st, a, b)
        if lang == 'plaintext': code.stylize(c)
        row = Text(k, style=c) + code
        if k != ' ': row.stylize(f"on {_mix(G['bg0'], c, .15)}")
        rows.append(row)
    return rows

def _hunk_end(ls, i, m):
    "Index past the hunk body that starts at `ls[i]`, from the counts in its header match `m`."
    o, n, j = int(ifnone(m[1], 1)), int(ifnone(m[2], 1)), i
    while j < len(ls) and (o > 0 or n > 0 or ls[j][:1] == '\\'):
        k = ls[j][:1]
        if k not in ('', ' ', '+', '-', '\\'): break
        o, n, j = o - (k in ('', ' ', '-')), n - (k in ('', ' ', '+')), j + 1
    return j

def is_diff(s):
    "Whether `s` holds a unified diff: at least one hunk header."
    return any(_HUNK.match(l) for l in (s or '').split('\n'))

def diff_lang(path):
    "fastpylight's language for a diffed `path`; a notebook, or one cell of it as `nb.ipynb#id`, is Python."
    return 'python' if path.partition('#')[0].endswith('.ipynb') else fastpylight.guess('', path)

def _header_lang(l):
    "The language a `---` or `+++` line names, or None."
    if l.startswith(('--- ', '+++ ')) and (f := l[4:].split('\t')[0].strip()) != '/dev/null': return diff_lang(f)

def diff_toks(s, lang=None):
    "Tokens per line of unified diff `s`, relative to the line after its prefix; `None` for every line outside a hunk body. `lang` overrides the `---`/`+++` paths."
    ls, out, cur, i = s.split('\n'), [], ifnone(lang, 'plaintext'), 0
    while i < len(ls):
        l = ls[i]; i += 1
        if lang is None: cur = ifnone(_header_lang(l), cur)
        out.append(None)
        if not (m := _HUNK.match(l)): continue
        j = _hunk_end(ls, i, m)
        out += hunk_toks(ls[i:j], cur); i = j
    return out

def toks_html(code, toks):
    "`code` escaped, each token a span whose classes are its scope and every dotted prefix of it, as `fastpylight.theme_css` names them."
    def cls(k): ps = k.split('.'); return ' '.join('hl-' + '-'.join(ps[:n]) for n in range(1, len(ps) + 1))
    out, at = [], 0
    for a, b, k in toks:
        if a < at: continue
        out += [html.escape(code[at:a]), f'<span class="{cls(k)}">{html.escape(code[a:b])}</span>']
        at = b
    return ''.join(out) + html.escape(code[at:])

def diff_html(s, lang=None):
    "Per line of unified diff `s`, the HTML of its code after the prefix, or `None` outside a hunk body."
    return [None if t is None else toks_html(l[1:], t) for l, t in zip(s.split('\n'), diff_toks(s, lang))]

def diff_rich(s, palette, theme='github_dark'):
    "`s` coloured in `palette` when it holds a unified diff, its code by the file's language, or a diffstat; else gray."
    G, ls = palette, s.split('\n')
    hunks = is_diff(s)
    if not hunks and not any(_STAT.match(l) for l in ls): return Text(s, style=G['gray'])
    rows, lang, i = [], 'plaintext', 0
    while i < len(ls):
        l = ls[i]; i += 1
        if hunks: lang = ifnone(_header_lang(l), lang)
        if hunks and l.startswith(('diff --git', '--- ', '+++ ')): rows.append(Text(l, style=f"bold {G['fg1']}"))
        elif hunks and (m := _HUNK.match(l)):
            rows.append(Text(l, style=G['aqua']))
            j = _hunk_end(ls, i, m)
            rows += _hunk(ls[i:j], lang, G, theme); i = j
        elif (m := _STAT.match(l)): rows.append(Text.assemble((m[1], G['gray']), (m[2], G['green']), (m[3], G['red'])))
        else: rows.append(Text(l, style=G['gray']))
    return Text('\n').join(rows)

In [ ]:
test_eq([diff_lang(p) for p in ('a/x.py', 'b/nbs/05_cli.ipynb', 'b/nbs/05_cli.ipynb#2874a64d', 'a/README.md')],
        ['python', 'python', 'python', 'markdown'])

In [ ]:
d = '--- a/x.py\n+++ b/x.py\n@@ -1,3 +1,4 @@\n import os\n-x = "a<b"\n+s = """one\n+two"""\n é = 1  # ü\n\\ No newline at end of file'
h = diff_html(d)
test_eq(len(h), len(d.split('\n')))
test_eq(h[:3] + h[-1:], [None] * 4)
assert '&lt;' in h[4] and 'hl-string' in h[6]
assert '# ü' in h[7] and h[7].startswith('<span class="hl-variable">é</span>')
test_eq(diff_html('@@ -1 +1 @@\n-a<\n+b', 'plaintext'), [None, 'a&lt;', 'b'])

The cell below colours a notebook cell's diff, as `edit_cell` reports it, with the `github-dark` theme and the palette dict `dark`. The added line has a green `+` mark, a green-tinted background and the token colours of its code. A string with no diff and no diffstat, such as `3 passed`, comes back as gray text with no spans:

In [ ]:
dark = {'fg1': '#c9d1d9', 'gray': '#8b949e', 'red': '#ff7b72', 'green': '#3fb950', 'aqua': '#39c5cf', 'bg0': '#0a0c10'}
nb_diff = diff_rich('--- a/nbs/x.ipynb#c1\n+++ b/nbs/x.ipynb#c1\n@@ -1 +1 @@\n-x = 1\n+def f(): return 2', dark)
added = nb_diff.split('\n')[-1]
test_eq(added.plain, '+def f(): return 2')
assert len({str(s.style) for s in added.spans}) > 2
test_eq(diff_rich('3 passed', dark).spans, [])
nb_diff